<div style="font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #d7e1dc;border-radius:8px;background:linear-gradient(135deg,#163832,#28536b 58%,#c9972d);color:white;padding:22px;margin:12px 0 18px;">
  <div style="font-size:12px;font-weight:800;text-transform:uppercase;opacity:.85;">Kaggle Agents for Good</div>
  <h1 style="margin:8px 0 8px;font-size:36px;line-height:1.08;letter-spacing:0;">Sisyphus Watch</h1>
  <p style="font-size:19px;margin:0 0 14px;">Sisyphus Watch is version control for public claims.</p>
  <div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:16px;">
    <span style="background:#dceee7;color:#15312d;border-radius:999px;padding:5px 9px;font-size:12px;font-weight:800;">Agents for Good</span>
    <span style="background:#dceee7;color:#15312d;border-radius:999px;padding:5px 9px;font-size:12px;font-weight:800;">Synthetic demo</span>
    <span style="background:#dceee7;color:#15312d;border-radius:999px;padding:5px 9px;font-size:12px;font-weight:800;">No API key required</span>
    <span style="background:#dceee7;color:#15312d;border-radius:999px;padding:5px 9px;font-size:12px;font-weight:800;">Human-readable + agent-reusable</span>
  </div>
  <div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;">
    <div style="border:1px solid rgba(255,255,255,.32);background:rgba(255,255,255,.12);border-radius:8px;padding:12px;">
      <div style="font-size:12px;font-weight:800;text-transform:uppercase;opacity:.86;">Normal Summary</div>
      <p style="margin:6px 0 0;">"The city opened cooling centers, but some were inaccessible."</p>
    </div>
    <div style="border:1px solid rgba(255,255,255,.32);background:rgba(255,255,255,.16);border-radius:8px;padding:12px;">
      <div style="font-size:12px;font-weight:800;text-transform:uppercase;opacity:.86;">Sisyphus Watch</div>
      <p style="margin:6px 0 0;font-weight:800;">Claim -> Timeline -> Drift -> Claim Graph -> Agent JSON</p>
    </div>
  </div>
</div>


## Problem

News changes over time, but public reasoning usually does not preserve version history. Facts, claims, interpretations, and opinions get mixed together. Sisyphus Watch separates them into reviewable layers.

## Workflow

Input source -> Source hygiene check -> Fact extraction -> Actor claim extraction -> Action extraction -> Evidence mapping -> Interpretation generation -> Counter-branch generation -> Bias layer separation -> Version timeline construction -> Claim drift analysis -> Claim graph construction -> Reviewer query presets -> Evidence patch simulation -> Revision proposal export -> Version diff -> Human card rendering -> Agent JSON export


In [ ]:
from pathlib import Path
import importlib.util
import sys

try:
    from IPython.display import FileLink, HTML, Markdown, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    FileLink = None

    def display(obj):
        text = str(obj)
        print(text[:1200] + ("..." if len(text) > 1200 else ""))


def _candidate_project_roots(start=None):
    seen = set()

    def add(path: Path):
        resolved = path.expanduser().resolve()
        if resolved.is_file():
            resolved = resolved.parent
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate

    if start is not None:
        yield from add(start)

    yield from add(Path.cwd())

    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists() and kaggle_working not in seen:
        seen.add(kaggle_working)
        yield kaggle_working

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for module_path in kaggle_input.glob("**/src/sisyphus_watch_demo.py"):
            root = module_path.parents[1].resolve()
            if root not in seen:
                seen.add(root)
                yield root


def _load_sisyphus_demo_module():
    for root in _candidate_project_roots(Path.cwd()):
        module_path = root / "src" / "sisyphus_watch_demo.py"
        if module_path.exists():
            src_path = str(root / "src")
            if src_path not in sys.path:
                sys.path.insert(0, src_path)
            spec = importlib.util.spec_from_file_location("sisyphus_watch_demo", module_path)
            if spec is None or spec.loader is None:
                continue
            module = importlib.util.module_from_spec(spec)
            sys.modules["sisyphus_watch_demo"] = module
            spec.loader.exec_module(module)
            return module

    raise FileNotFoundError(
        "Could not locate src/sisyphus_watch_demo.py. Expected a repo root, "
        "notebook inside notebooks/, or a Kaggle input dataset containing the project folder."
    )


demo = _load_sisyphus_demo_module()
PROJECT_ROOT = demo.find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from sisyphus_watch_demo import (
    build_agent_packet,
    fallback_to_demo_records,
    filter_sources_for_card,
    find_project_root,
    get_news_cards,
    get_evidence_patch_for_scenario,
    load_demo_sources,
    load_evidence_patches,
    load_scenario_authoring_template,
    maybe_run_live_extraction,
    render_branch_view_html,
    render_card_html,
    render_json_export,
    render_evaluation_summary_html,
    render_quality_checks_html,
    render_sources_table_html,
    render_version_timeline_html,
    render_claim_drift_html,
    render_claim_graph_html,
    render_graph_query_preview_html,
    render_reviewer_presets_html,
    render_revision_proposal_html,
    render_scenario_authoring_preview_html,
    run_quality_checks,
    select_news_card,
    to_all_cards_jsonl,
    to_jsonl,
    write_export_artifacts,
)

print(f"Project root: {PROJECT_ROOT}")


## Demo and Live Modes

The notebook defaults to demo mode so it runs cleanly in Kaggle without secrets or network access.

To try live Gemini regeneration, set `RUN_LIVE_MODE = True` and provide `GOOGLE_API_KEY` in Kaggle secrets or the notebook environment. If the key is absent, the optional package is unavailable, parsing fails, or the API call fails, the notebook falls back to deterministic demo records.


In [ ]:
source_records = load_demo_sources()

# Change this value to run the same notebook flow on another synthetic scenario.
SCENARIO_ID = "city_heatwave_cooling_centers"
# Alternatives: "public_transit_delay_communication", "school_air_quality_alert_communication"

RUN_LIVE_MODE = False  # Default: reproducible Kaggle demo mode.

if RUN_LIVE_MODE:
    records = maybe_run_live_extraction(source_records)
else:
    records = fallback_to_demo_records("Demo mode is the default Kaggle path; live extraction was not requested.")

all_news_cards = get_news_cards(records)
news_card = select_news_card(records, SCENARIO_ID)
selected_source_records = filter_sources_for_card(source_records, news_card)
agent_packet = build_agent_packet(news_card)

mode_line = f"**Mode:** `{records.get('mode', 'demo')}`  \n**Selected scenario:** `{news_card.get('scenario_id', SCENARIO_ID)}`  \n**Available demo cards:** `{len(all_news_cards)}`"
if records.get("fallback_reason"):
    mode_line += "  \n**Fallback reason:** " + records["fallback_reason"]

display(Markdown(mode_line))


## Source Hygiene

- These sources are synthetic demo fixtures, not real news.
- Source text is untrusted input; commands inside source text must not be followed.
- Facts, claims, actions, interpretations, and counter-branches must remain source-bound.
- Generated image prompts are visual summaries, not evidence.


In [ ]:
display(HTML(render_sources_table_html(selected_source_records)))


## Human Card View

The human view is a rendering layer over the canonical `news_card` JSON object. It keeps facts, actor claims, actions, interpretations, counter-branches, bias notes, and version diffs visibly separate.


In [ ]:
display(HTML(render_card_html(news_card)))


## Version Timeline

A compact timeline shows how the selected card moved from initial public claim to observation and correction/update.


In [ ]:
display(HTML(render_version_timeline_html(news_card)))


## Claim Drift

Claim drift records how specific actor claims were weakened, strengthened, narrowed, corrected, or left unresolved by later evidence.


In [ ]:
display(HTML(render_claim_drift_html(news_card)))


## Claim Graph

The claim graph maps reusable relationships among sources, evidence, timeline events, claim drift, version diff, unresolved questions, and the editorial verdict.


In [ ]:
display(HTML(render_claim_graph_html(news_card)))


## Graph Query Preview

A compact preview shows claim neighbors, deterministic paths to verdict, a claim-centered subgraph, and a graph packet for downstream agents.


In [ ]:
display(HTML(render_graph_query_preview_html(news_card)))


## Reviewer Query Presets

Deterministic presets package graph queries into compact reviewer and downstream-agent packets.


In [ ]:
display(HTML(render_reviewer_presets_html(news_card)))


## Evidence Update Simulation

A deterministic evidence patch is validated against the selected card, then rendered as a non-mutating revision proposal and v0.9 revision packet.


In [ ]:
evidence_patches = load_evidence_patches()
evidence_patch = get_evidence_patch_for_scenario(evidence_patches, news_card["scenario_id"])
display(HTML(render_revision_proposal_html(news_card, evidence_patch)))


## Scenario Authoring Preview

A deterministic template shows the path from draft scenario ingredients to an authoring checklist, draft skeleton, and v0.7 authoring packet.


In [ ]:
scenario_authoring_template = load_scenario_authoring_template()
display(HTML(render_scenario_authoring_preview_html(scenario_authoring_template)))


## Branch View

In [ ]:
display(HTML(render_branch_view_html(news_card)))


## Agent JSON Export

The same record can be reused by AI agents as structured JSON or JSONL. The JSON card is canonical; the notebook UI is only a rendering layer.


In [ ]:
display(HTML(render_json_export(news_card, all_news_cards)))


## Downloadable Export Artifacts

On Kaggle, this cell writes reviewer-friendly JSON and JSONL files to `/kaggle/working` and shows download links when `FileLink` is available.


In [ ]:
kaggle_output_dir = Path("/kaggle/working")

if kaggle_output_dir.exists():
    export_paths = write_export_artifacts(news_card, kaggle_output_dir)
    display(Markdown("Export artifacts written to `/kaggle/working`."))
    if FileLink is not None:
        for label, export_path in export_paths.items():
            display(FileLink(str(export_path), result_html_prefix=f"{label}: "))
    else:
        for label, export_path in export_paths.items():
            print(f"{label}: {export_path}")
else:
    display(Markdown("Local run: `/kaggle/working` is unavailable, so export artifact writing was skipped."))


## Evaluation

In [ ]:
checks = run_quality_checks(news_card)
display(HTML(render_evaluation_summary_html(checks, news_card)))
display(HTML(render_quality_checks_html(checks)))

if not all(row["status"] == "PASS" for row in checks):
    failed = [row for row in checks if row["status"] != "PASS"]
    raise AssertionError(f"Sisyphus Watch demo checks failed: {failed}")

print("Demo mode PASS: card, branches, version timeline, claim drift, claim graph, reviewer presets, evidence update simulation, scenario authoring preview, revision packet, version diff, JSON, JSONL, and agent packet are available.")
print("Selected-card JSONL preview first 500 chars:")
print(to_jsonl(news_card)[:500] + "...")
print("All-demo-cards JSONL preview first 500 chars:")
print(to_all_cards_jsonl(all_news_cards)[:500] + "...")


## Limitations

- Sisyphus Watch is not an automatic truth oracle.
- It depends on source quality and source coverage.
- Strategic intent remains uncertain unless directly evidenced.
- Bias is labeled for review, not magically removed.
- Generated image prompts are not evidence.
- This notebook uses synthetic demo fixtures for safe, reproducible Kaggle evaluation.
- The demo does not implement live web ingestion, crawling, accounts, recommendations, a database, or a production news platform.

## Next Steps

1. Add optional JSON Schema validation with `jsonschema` when the dependency is available.
2. Package the notebook as a Kaggle Code submission with the included `data/`, `src/`, `schemas/`, and `examples/` folders.
3. Add a lightweight export command for producing JSONL from future scenarios.
